# Secure Aggregation (SecAgg+): Concepts and Educational Simulation

**Purpose.** Explain the role of secure aggregation in federated learning and demonstrate, with a **clearly labeled non-secure educational simulation**, how additive masking can hide individual client updates while revealing only their sum. We also validate the Flower `SecAggPlusWorkflow` parameterization for Flower 1.29+ and discuss dropout/threshold secret sharing.

> **Important:** The masking demo below is for teaching only. It is **NOT** production cryptography, does not implement the full SecAgg+ protocol, and provides no formal security guarantee.

## Learning objectives
- Understand what SecAgg+ protects (individual updates from the server) and what it does not (DP, model inversion guarantees).
- Instantiate/check `flwr.server.workflow.SecAggPlusWorkflow` parameters.
- Simulate pairwise additive masks and verify unmasked-sum correctness; show why dropout breaks naive pairwise masking.
- Explain threshold secret sharing intuition and communication/robustness trade-offs.

## Background and math

### Goal
Clients hold updates $u_i$. The server should learn $S=\sum_i u_i$ without learning any individual $u_i$.
*Meaning here:* aggregation protects per-client contributions in transit/at server, complementing (not replacing) differential privacy.

### Pairwise additive masking (teaching sketch)
For each pair $(i,j)$, sample mask $m_{ij}$ and set $m_{ji}=-m_{ij}$. Client $i$ sends $\tilde u_i = u_i + \sum_{j\ne i} m_{ij}$ (mod $M$ in quantized form). Masks cancel:
$$
\sum_i \tilde u_i = \sum_i u_i.
$$
*Meaning here:* individual masked updates can look random, but the aggregate is preserved.

### Dropout and secret sharing
If a client drops after masking, its masks do not cancel. Threshold secret sharing (e.g. Shamir, reconstruct with $t$ of $n$ shares) lets remaining clients help unmask dropped users' contributions securely in the full protocol.
*Meaning here:* robustness to dropouts requires additional protocol rounds/cryptography beyond pairwise masks.

## Assumptions and limitations
- Educational simulation uses integer quantization and modular arithmetic; no key agreement, authentication, or malicious-security proofs.
- Flower SecAgg+ API/behavior depends on installed version; we validate parameter construction only.
- No committed end-to-end SecAgg run log is claimed unless present in artifacts.
- **Repo/runtime status (D-PRIV-004):** protocol-level tests/parameterization exist, but end-to-end Flower simulation orchestration via `server_app.main` is not verified in this repo and fails closed with `NotImplementedError` when `secagg-enabled=true` (no hang, no plaintext fallback). Do not claim end-to-end SecAgg+ Flower-app runtime evidence from this notebook without a verified committed run log.
- **Server visibility:** SecAgg+ protects parameter updates; per-client metrics (e.g. client IDs, train/eval metrics) and sample counts can remain visible to the server/strategy in Flower's legacy-compat workflow path.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    n_clients: int = 4
    dim: int = 16 if nb.MODE == "light" else 64
    mod_bits: int = 32
cfg = Cfg(); cfg


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Check Flower SecAggPlusWorkflow (installed API)
secagg_ok = False
wf = None
secagg_note = None
try:
    from flwr.server.workflow import SecAggPlusWorkflow
    wf = SecAggPlusWorkflow(
        num_shares=4,
        reconstruction_threshold=3,
        max_weight=1000.0,
        clipping_range=8.0,
        quantization_range=2**22,
    )
    secagg_ok = True
    secagg_note = 'SecAggPlusWorkflow instantiated successfully'
    print(secagg_note, type(wf))
    print('num_shares=', getattr(wf, 'num_shares', None),
          'reconstruction_threshold=', getattr(wf, 'reconstruction_threshold', None))
except Exception as e:
    secagg_note = f'[fallback] SecAggPlusWorkflow unavailable/incompatible: {e}'
    print(secagg_note)


## Protocol overview (diagrams)

In [ ]:
# Figure 1: sequence / flow as text-rendered diagram via matplotlib (Mermaid shown in markdown)
fig, ax = plt.subplots(figsize=(9,4.5))
ax.axis('off')
steps = [
    '1. Setup: clients agree on keys/parameters',
    '2. Advertise keys / share metadata (server-mediated)',
    '3. Masked upload: each client sends masked quantized update',
    '4. Consistency/unmask: server requests shares to cancel masks',
    '5. Aggregate: server recovers sum only (not individual updates)',
]
for i,s in enumerate(steps):
    ax.text(0.02, 0.85 - i*0.17, s, fontsize=11, va='top', family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#eef6fb', edgecolor='#2a6f97'))
ax.set_title('SecAgg+ (high-level stages) — schematic')
fig.tight_layout(); nb.save_fig(fig, 'security_secagg_sequence'); plt.show()


**How to read this / what it shows.** Schematic stages of secure aggregation: individual updates are masked; only the aggregate should be recoverable by the server when the protocol completes correctly.

In [ ]:
# Mermaid diagrams (rendered by notebook frontends that support Mermaid)
mermaid_seq = r"""
```mermaid
sequenceDiagram
    participant C1 as Client 1
    participant C2 as Client 2
    participant S as Server
    C1->>S: masked update ũ1
    C2->>S: masked update ũ2
    S->>C1: unmask info / share requests
    S->>C2: unmask info / share requests
    C1->>S: shares for recovery
    C2->>S: shares for recovery
    S->>S: compute sum(u1+u2), not u1,u2
```
"""
mermaid_pki = r"""
```mermaid
flowchart LR
    CA[CA cert] --> S[Server cert]
    CA --> C[Client certs]
    S <--> |mTLS| C
```
"""
print('Mermaid sequence diagram (markdown cell below in some viewers):')
print(mermaid_seq)


```mermaid
sequenceDiagram
    participant C1 as Client 1
    participant C2 as Client 2
    participant S as Server
    C1->>S: masked update ũ1
    C2->>S: masked update ũ2
    S->>C1: unmask/share requests
    S->>C2: unmask/share requests
    C1->>S: recovery shares
    C2->>S: recovery shares
    S->>S: recover sum(u1+u2)
```

**How to read this / what it shows.** Message flow intuition: masks cancel in aggregate; dropout handling requires additional recovery shares (threshold secret sharing in full SecAgg+).


## Educational simulation (NOT secure, teaching only)

In [ ]:
# Pairwise additive masks in modular arithmetic (teaching only)
MOD = 2**cfg.mod_bits
SCALE = 2**20

def masked_updates(updates, rng, mod=MOD, scale=SCALE):
    updates = np.asarray(updates, dtype=np.float64)
    n = len(updates)
    q = np.rint(updates * scale).astype(np.int64) % mod
    masks = np.zeros((n, updates.shape[1]), dtype=np.int64)
    for i in range(n):
        for j in range(i + 1, n):
            m = rng.integers(0, mod, size=updates.shape[1], dtype=np.int64)
            masks[i] = (masks[i] + m) % mod
            masks[j] = (masks[j] - m) % mod
    masked = (q + masks) % mod
    return q, masked, masks, scale

def unmask_sum(masked, scale=SCALE, mod=MOD):
    s = masked.sum(axis=0) % mod
    # signed decode
    s = np.where(s > mod // 2, s - mod, s)
    return s.astype(np.float64) / scale

# toy client updates
true_updates = rng.normal(0, 0.5, size=(cfg.n_clients, cfg.dim))
q, masked, masks, scale = masked_updates(true_updates, np.random.default_rng(nb.SEED))
plain_sum = true_updates.sum(axis=0)
plain_sum_q = np.rint(plain_sum * scale) / scale
rec_sum = unmask_sum(masked, scale=scale)
qerr = float(np.max(np.abs(rec_sum - plain_sum_q)))
print('max quantization-consistent reconstruction error:', qerr)
assert np.allclose(rec_sum, plain_sum_q, atol=1e-6), 'unmasked sum mismatch'
print('[ok] unmasked sum equals quantized plain sum (teaching simulation)')


In [ ]:
# Figure 2: plain vs masked distribution (masked should look spread/uniform-ish over integers)
fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].hist(true_updates.ravel(), bins=30, alpha=0.7, color='#2a6f97')
axes[0].set_title('Plain update values (example)'); axes[0].set_xlabel('Value'); axes[0].set_ylabel('Count')
axes[1].hist((masked.ravel() % MOD), bins=40, alpha=0.7, color='#d1495b')
axes[1].set_title('Masked values (mod 2^32 space, teaching)'); axes[1].set_xlabel('Masked integer'); axes[1].set_ylabel('Count')
fig.tight_layout(); nb.save_fig(fig, 'security_secagg_masked_vs_plain'); plt.show()


**How to read this / what it shows.** Individual masked encodings are transformed into a different numeric domain; without unmasking secrets, single-client values are not directly readable from masked integers in this toy setup. (Not a security proof.)

In [ ]:
# Figure 3: mask cancellation visualization and dropout failure
mask_sum = masks.sum(axis=0) % MOD
assert np.all(mask_sum == 0), 'pairwise masks should cancel in aggregate'
# dropout: remove client 0 after masking
masked_drop = masked.copy()
masked_drop[0] = 0  # dropped contribution missing
broken = unmask_sum(masked_drop, scale=scale)
dropout_max_abs_error = float(np.max(np.abs(broken - plain_sum_q)))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(mask_sum[:min(32, len(mask_sum))], marker='o')
axes[0].set_title('Sum of pairwise masks mod M (should be 0)')
axes[0].set_xlabel('Coordinate (first 32)')
axes[0].set_ylabel('Value mod M')
axes[1].plot(plain_sum_q[:min(32, len(plain_sum_q))], label='true quantized sum', marker='o')
axes[1].plot(broken[:min(32, len(broken))], label='reconstructed with dropout (broken)', marker='x')
axes[1].set_title('Dropout breaks naive pairwise unmasking')
axes[1].set_xlabel('Coordinate')
axes[1].set_ylabel('Sum value')
axes[1].legend()
fig.tight_layout()
nb.save_fig(fig, 'security_secagg_dropout')
plt.show()
print('max abs error with dropout:', dropout_max_abs_error)
assert dropout_max_abs_error > 1e-6, 'dropout should break naive reconstruction in this demo'


**How to read this / what it shows.** Left: pairwise masks cancel in aggregate. Right: if a client drops after contributing its masked value, naive reconstruction is incorrect—motivating threshold recovery in full SecAgg+.

## Threshold secret sharing intuition (small demo)

In [ ]:
# Tiny Shamir-like intuition over a prime field (illustrative, NOT full SecAgg+)
# Share secret s as y_i = s + a*x_i mod p; recover with >=2 points via Lagrange interpolation at 0.
p = 2**31 - 1

def modinv(a, p):
    return pow(a, p - 2, p)

def share_secret(s, n, t, rng):
    coeffs = [s % p] + [int(rng.integers(1, p)) for _ in range(t - 1)]
    xs = list(range(1, n + 1))
    ys = []
    for x in xs:
        y = 0
        for k, c in enumerate(coeffs):
            y = (y + c * pow(x, k, p)) % p
        ys.append((x, y))
    return ys

def reconstruct(points):
    # Lagrange at x=0
    s = 0
    for j, (xj, yj) in enumerate(points):
        num, den = 1, 1
        for m, (xm, _) in enumerate(points):
            if m == j:
                continue
            num = (num * (-xm % p)) % p
            den = (den * ((xj - xm) % p)) % p
        s = (s + yj * num * modinv(den, p)) % p
    return s

s = 123456789
shares = share_secret(s, n=4, t=3, rng=np.random.default_rng(nb.SEED))
rec3 = reconstruct(shares[:3])
rec2 = reconstruct(shares[:2])  # wrong threshold (insufficient shares)
print('secret:', s, 'reconstruct with 3:', rec3, 'ok:', rec3 == s, 'reconstruct with 2 equals secret?', rec2 == s)
assert rec3 == s, 'threshold reconstruction with sufficient shares should succeed'


In [ ]:
# Figure 4: communication overhead and threshold robustness (analytic)
clients_n = np.array([2,4,8,16,32])
pairwise_msgs = clients_n*(clients_n-1)  # pairwise key/mask exchanges scale ~O(n^2)
dropout = np.linspace(0, 0.5, 11)
fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].bar(clients_n.astype(str), pairwise_msgs, color='#00798c')
axes[0].set_title('Analytic pairwise exchange count ~ O(n²)'); axes[0].set_xlabel('Clients (n)'); axes[0].set_ylabel('Pairwise interactions (illustrative)')
# success rate toy: succeed if surviving >= threshold
for t in [2,3,4]:
    n=4
    # approximate survival probability with independent dropouts
    from math import comb
    succ = []
    for d in dropout:
        prob = sum(comb(n,k)*((1-d)**k)*(d**(n-k)) for k in range(t,n+1))
        succ.append(prob)
    axes[1].plot(dropout, succ, marker='o', label=f'n=4, threshold={t}')
axes[1].set_title('Illustrative completion probability vs dropout rate'); axes[1].set_xlabel('Dropout rate'); axes[1].set_ylabel('P(survive >= threshold)'); axes[1].set_ylim(0,1.05); axes[1].legend()
fig.tight_layout(); nb.save_fig(fig, 'security_secagg_overhead_dropout'); plt.show()


**How to read this / what it shows.** Left: pairwise coordination grows quickly with client count (schematic). Right: higher thresholds require more survivors, reducing dropout tolerance for fixed n.

In [ ]:
# Figure 5: quantization error vs scale (teaching)
scales = [2**10, 2**14, 2**18, 2**22]
errs = []
for sc in scales:
    _, msk, _, _ = masked_updates(true_updates, np.random.default_rng(nb.SEED), mod=MOD, scale=sc)
    rec = unmask_sum(msk, scale=sc)
    errs.append(np.max(np.abs(rec - np.rint(true_updates.sum(axis=0)*sc)/sc)))
fig, ax = plt.subplots(figsize=(6.5,4))
ax.plot(scales, errs, marker='o')
ax.set_xscale('log', base=2); ax.set_title('Quantization-consistent reconstruction error vs scale'); ax.set_xlabel('Scale (log2)'); ax.set_ylabel('Max abs error')
fig.tight_layout(); nb.save_fig(fig, 'security_secagg_quantization'); plt.show()
pd.DataFrame({'scale': scales, 'max_abs_error': errs})


**How to read this / what it shows.** Larger quantization scales reduce representational error of real-valued updates in this toy encoding; practical protocols choose ranges to balance precision and overflow/security parameters.

## Artifacts and evidence

In [ ]:
secagg_log = nb.ARTIFACTS / 'secagg_run.log'
if secagg_log.exists():
    txt = secagg_log.read_text(errors='ignore')
    print('Parsed artifact: artifacts/secagg_run.log (first 40 lines)')
    print('\n'.join(txt.splitlines()[:40]))
else:
    print('[evidence] No committed end-to-end SecAgg Flower-app run log found (artifacts/secagg_run.log missing). server_app.main(secagg-enabled=true) fails closed (NotImplementedError); see D-PRIV-006.')
    print('Protocol-level coverage: tests/integration/test_secagg.py; fail-closed runtime guard: tests/unit/fl/test_server_app.py.')


## Quantitative summary


In [ ]:
summary = pd.DataFrame([
    {'check': 'secaggplus_workflow_instantiated', 'value': secagg_ok},
    {'check': 'secaggplus_workflow_note', 'value': secagg_note},
    {'check': 'num_clients', 'value': cfg.n_clients},
    {'check': 'vector_dim', 'value': cfg.dim},
    {'check': 'pairwise_mask_cancel_ok', 'value': bool(np.all(mask_sum == 0))},
    {'check': 'reconstruction_error_quantized_sum', 'value': qerr},
    {'check': 'dropout_breaks_naive_reconstruction', 'value': bool(dropout_max_abs_error > 1e-6)},
    {'check': 'dropout_max_abs_error', 'value': dropout_max_abs_error},
])
summary


## Sensitivity / ablation
- **Quantization scale/range:** too small loses precision; too large risks overflow in fixed modular domains.
- **Threshold $t$ vs dropouts:** higher $t$ improves security margins in some designs but reduces liveness.
- **Client count:** coordination and communication costs grow; protocol choice matters at scale.

## Limitations and failure cases
- Simulation omits authentication, key agreement, malicious clients/server, and full SecAgg+ message schedule.
- Masked distributions in toy integer space are illustrative, not proofs of indistinguishability.
- Secure aggregation alone does not provide differential privacy.

## Conclusions and further experiments
- SecAgg+ aims to reveal only aggregates, protecting individual updates from an honest-but-curious server under protocol assumptions.
- Correctness of mask cancellation and the need for dropout recovery were demonstrated in a teaching simulation.
- Further work: run end-to-end Flower SecAgg+ with logs, combine with DP accounting, and evaluate overhead under realistic network conditions.

## References
- K. Bonawitz et al. "Practical Secure Aggregation for Privacy-Preserving Machine Learning." *CCS*, 2017.
- J. H. Bell et al. "Secure Single-Server Aggregation with (Poly)Logarithmic Overhead." *CCS*, 2020. (SecAgg+ context)
- Flower documentation: Secure Aggregation workflows (installed version behavior applies).
